# OSP — DOTA training run

Trains the Orbital Scene Preprocessor detector on **real aerial imagery** instead of
the synthetic shapes `data/synth_demo.py` draws.

## Before you run

In the panel on the right:

1. **Accelerator** -> `GPU T4` (single) or `P100`. Without a GPU the run will not
   finish. Do NOT pick `T4 x2` — `model/train_6ch.py` trains on a single CUDA
   device with no `DataParallel`/DDP wrapping, so a second GPU sits idle and you
   would just burn quota twice as fast for no speedup.
2. **Internet** -> `On`. The notebook clones GitHub and downloads DOTA.
3. **Persistence** -> optional, but handy if you want to resume.

Then `Run All` and leave it.

**Budget the time from cell 4a, not from a guess.** This run is loader-bound:
measured on a 4-core box, the tile pipeline sustains about 29 tiles/s, which
puts a 20,000-tile corpus at roughly 12 minutes per epoch and the default
8+40 epochs near 9 hours. That fits a session, with little margin for the
DOTA download and tiling ahead of it. Cell 4a measures the real numbers on
the machine you actually got; if the projection it prints is uncomfortable,
lower `EPOCHS_PHASE2` before starting rather than losing the run to a timeout.

## What comes out

A single file, `osp_dota_artifacts.zip`, in the notebook output. Download it,
unzip into the repo, and the rest of the pipeline (INT8 export, briefs, benchmarks)
runs locally on your Mac.

## Set SMOKE = True for the first run

It processes 40 source images and trains 2+2 epochs, in about 15 minutes.
Confirms every stage works before you commit hours of GPU time to it.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

SMOKE = True     # True: ~15 min end-to-end check. False: the real run.

REPO   = 'https://github.com/brightyorcerf/orbital-preprocessor'
BRANCH = 'main'

# Training. Phase 1 trains the new stem with the backbone frozen; phase 2
# unfreezes and trains the whole network at a lower rate.
EPOCHS_PHASE1 = 2 if SMOKE else 8
EPOCHS_PHASE2 = 2 if SMOKE else 40
BATCH         = 16 if SMOKE else 32

# Dataloader workers. This is not a minor knob on this run. Tiles are stored as
# RGB JPEG and the six bands are derived on read, which costs tens of
# milliseconds of OpenCV per tile against roughly 10 ms for a yolov8n step on a
# T4. At WORKERS=2 the GPU spends most of the run waiting for tiles. A Kaggle
# T4 session has 4 vCPUs, so 4 is the number; cell 9a measures whether it is
# actually keeping up before you commit the hours.
WORKERS       = 4

# Tiles used for the per-epoch validation that selects the best checkpoint.
# Validation now batches its forward pass and reads through the same worker
# pool, so a useful number of tiles is affordable. The full val split is scored
# once at the end regardless.
VAL_LIMIT = 32 if SMOKE else 600
VAL_BATCH = 16

# Tiling. --limit caps source images per split; None means all of them.
LIMIT = 40 if SMOKE else None

print(f'SMOKE={SMOKE}  phase1={EPOCHS_PHASE1}  phase2={EPOCHS_PHASE2}  '
      f'batch={BATCH}  workers={WORKERS}  val_limit={VAL_LIMIT}  limit={LIMIT}')


## 1. Confirm the GPU is actually attached

This cell stops the notebook if it is not. The custom training loop in
`model/train_6ch.py` will silently fall back to CPU otherwise, and you would
discover that several wasted hours later.


In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), (
    'No CUDA device. Set Accelerator to GPU in the right-hand panel and restart.'
)
print('torch', torch.__version__, '| device:', torch.cuda.get_device_name(0))


## 2. Clone the repository

The notebook deliberately uses the real repo rather than a copy pasted into
this page, so the code that trains here is the code that is committed.


In [ ]:
import os, shutil, pathlib

WORK = pathlib.Path('/kaggle/working')
TEMP = pathlib.Path('/kaggle/temp')      # scratch; not counted against output quota
TEMP.mkdir(parents=True, exist_ok=True)

SRC = WORK / 'orbital-preprocessor'
if SRC.exists():
    shutil.rmtree(SRC)

!git clone --depth 1 --branch {BRANCH} {REPO} {SRC}
os.chdir(SRC)
print('cwd:', os.getcwd())


In [ ]:
# Kaggle already carries torch, numpy and opencv. Ultralytics supplies the
# YOLOv8 model definition that stem_swap.py operates on.
!pip install -q ultralytics==8.4.125 2>&1 | tail -2
import ultralytics; print('ultralytics', ultralytics.__version__)


## 3. Download DOTA-v1.0

About 2 GB, from the Ultralytics asset mirror. It unpacks to:

```
DOTAv1/
  images/{train,val}/*.png
  labels/{train,val}/*.txt              normalised oriented quads
  labels/{train,val}_original/*.txt     original DOTA annotations
```

`data/dota_prep.py` reads either label format. It uses `labels/{split}/`, which
is the normalised one.

Everything lands in `/kaggle/temp` so it does not consume the 20 GB output quota.


In [ ]:
DOTA_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/DOTAv1.zip'
DOTA_ZIP = TEMP / 'DOTAv1.zip'
DOTA_DIR = TEMP / 'dota'

if not (DOTA_DIR / 'DOTAv1' / 'images').exists():
    if not DOTA_ZIP.exists():
        !curl -L --retry 3 -o {DOTA_ZIP} {DOTA_URL}
    DOTA_DIR.mkdir(parents=True, exist_ok=True)
    !unzip -q -o {DOTA_ZIP} -d {DOTA_DIR}
    DOTA_ZIP.unlink(missing_ok=True)   # reclaim 2 GB immediately

DOTA_ROOT = DOTA_DIR / 'DOTAv1'
!ls {DOTA_ROOT} && ls {DOTA_ROOT}/images && echo '--- counts ---'
!ls {DOTA_ROOT}/images/train | wc -l && ls {DOTA_ROOT}/images/val | wc -l


## 4. Tile DOTA into OSP training tiles

This step does four things, each of which is a place the run could go quietly wrong,
so `data/dota_prep.py` reports on all of them:

- keeps only `ship`, `plane`, `storage tank`, `harbor`, discarding DOTA's other eleven categories
- flattens DOTA's **oriented** quads to axis-aligned boxes, and reports the mean area
  inflation this costs (a diagonal ship roughly doubles in box area)
- slices large scenes into 640x640 tiles with overlap, dropping any box that loses
  more than 65% of its area to the crop, so tiling cannot manufacture labels from slivers
- writes **RGB JPEGs**, not 6-band arrays. The six bands are derived at read time.
  A materialised 6-band float32 tile is 9.8 MB; this corpus would be ~200 GB that way,
  against roughly 2 GB as JPEG.


In [ ]:
LIMIT_ARG = f'--limit {LIMIT}' if LIMIT else ''
TILES = TEMP / 'osp_dota'

!python data/dota_prep.py --src {DOTA_ROOT} --out {TILES} {LIMIT_ARG}


In [ ]:
import json
manifest = json.loads((TILES / 'prep_manifest.json').read_text())
for split, s in manifest['splits'].items():
    print(f"{split:6s} {s['tiles_written']:6d} tiles  {s['instances']:7d} instances  "
          f"AABB inflation mean {s['aabb_mean_inflation']} p95 {s['aabb_p95_inflation']}")
    print('       ', s['per_class'])


### Look at a tile before training on 20,000 of them

Boxes drawn from the written label files, not from memory. If the tiling had a
coordinate bug, it is visible here and nowhere else until accuracy comes out wrong.


In [ ]:
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path

NAMES = manifest['classes']
lbls = [p for p in sorted((TILES/'labels'/'train').glob('*.txt')) if p.read_text().strip()]
random.seed(0)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, lf in zip(axes, random.sample(lbls, min(3, len(lbls)))):
    img = cv2.cvtColor(cv2.imread(str(TILES/'images'/'train'/(lf.stem+'.jpg'))), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    for row in lf.read_text().splitlines():
        c, cx, cy, bw, bh = row.split()
        cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
        p1 = (int(cx-bw/2), int(cy-bh/2)); p2 = (int(cx+bw/2), int(cy+bh/2))
        cv2.rectangle(img, p1, p2, (255, 60, 60), 2)
        cv2.putText(img, NAMES[int(c)], (p1[0], max(12, p1[1]-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 60, 60), 1)
    ax.imshow(img); ax.set_title(lf.stem, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()


## 4a. Is the GPU going to be fed?

This run is loader-bound, not compute-bound: deriving six bands from a JPEG costs
far more than a yolov8n step on a T4. The cell below times the real dataloader
against a real training step, so you find out now rather than five hours in.

If the loader is slower than the step, the GPU idles for the difference and the
whole run stretches by that ratio. Raise `WORKERS`, or cut `EPOCHS_PHASE2`, before
starting.


In [ ]:
import sys, time, torch
sys.path.insert(0, '.')
from ground.dataset_6ch import MultiSpectralDataset
from model.train_6ch import AugmentedTiles, yolo_collate, worker_init

_ds = AugmentedTiles(MultiSpectralDataset(TILES/'images'/'train',
                                          TILES/'labels'/'train', 640), seed=0)
_dl = torch.utils.data.DataLoader(
    _ds, batch_size=BATCH, shuffle=True, num_workers=WORKERS,
    collate_fn=yolo_collate, worker_init_fn=worker_init,
    persistent_workers=True, pin_memory=True, prefetch_factor=2, drop_last=True)

_it = iter(_dl); next(_it)                       # pay worker startup once
t = time.time(); N = 8
for _ in range(N): _b = next(_it)
load_s = (time.time() - t) / N

from model.train_6ch import load_or_create_model, build_optimizer, make_scaler
_m = load_or_create_model('model/artifacts/yolov8n_6ch.pt', 'yolov8n.pt', 4, 'cuda')
from ultralytics.utils.loss import v8DetectionLoss
_c = v8DetectionLoss(_m); _o = build_optimizer(_m, 1e-3); _s = make_scaler(True)
_x = _b['img'].cuda()
for _ in range(3):                               # warm up cudnn autotune
    with torch.autocast('cuda'):
        _l, _ = _c(_m(_x), _b)
    _s.scale(_l.sum()).backward(); _s.step(_o); _s.update(); _o.zero_grad()
torch.cuda.synchronize(); t = time.time()
for _ in range(N):
    with torch.autocast('cuda'):
        _l, _ = _c(_m(_x), _b)
    _s.scale(_l.sum()).backward(); _s.step(_o); _s.update(); _o.zero_grad()
torch.cuda.synchronize(); step_s = (time.time() - t) / N

epochs = EPOCHS_PHASE1 + EPOCHS_PHASE2
batches = len(_dl)
print(f'batch of {BATCH}:  load {load_s*1000:6.0f} ms   gpu step {step_s*1000:6.0f} ms')
print(f'bound by: {"DATALOADER" if load_s > step_s else "GPU"}   '
      f'utilisation ~{min(load_s, step_s)/max(load_s, step_s)*100:.0f}%')
print(f'{batches} batches/epoch x {epochs} epochs '
      f'-> ~{max(load_s, step_s)*batches*epochs/3600:.1f} h of training')
del _dl, _it, _m, _c, _o, _s


## 5. Train

`model/train_6ch.py` performs the stem swap (3 channels/80 classes -> 6 channels/4 classes)
and then runs its own two-phase loop. It does not call `YOLO().train()`, because
Ultralytics' data pipeline cannot read 6-channel float tiles.

Expect the number to land far below the synthetic model's 0.99. **That is the point
of this run.** A moderate score on real aerial imagery means something; a perfect
score on shapes the repository drew itself does not.


In [ ]:
import time
t0 = time.time()

# Mixed precision and the weight EMA are on by default on CUDA
# (--no-amp / --ema-decay 0 turn them off).
!python model/train_6ch.py \
    --dataset {TILES} \
    --device cuda \
    --epochs-phase1 {EPOCHS_PHASE1} \
    --epochs {EPOCHS_PHASE2} \
    --batch {BATCH} \
    --workers {WORKERS} \
    --val-limit {VAL_LIMIT} \
    --val-batch {VAL_BATCH} \
    --out model/artifacts/osp_best.pt \
    --metrics-out model/artifacts/train_metrics.json

print(f'\ntraining wall clock: {(time.time()-t0)/60:.1f} min')


In [ ]:
metrics = json.loads(Path('model/artifacts/train_metrics.json').read_text())
print(json.dumps(metrics, indent=2)[:2000])


## 6. Package the results

Only the checkpoint, the metrics and the tiling manifest come back. The tiles
themselves stay here: they are reproducible from `data/dota_prep.py` and a copy
of DOTA, so shipping gigabytes of them home would be pointless.

The validation tiles **do** come back, because the INT8 calibration and the
accuracy re-scoring you run locally have to use the same held-out split this
model was scored against. Calibrating on a different distribution than the one
the model runs on is the classic way to lose small-object recall silently.


In [ ]:
import shutil, os

OUT = WORK / 'osp_dota_out'
if OUT.exists(): shutil.rmtree(OUT)
(OUT / 'model' / 'artifacts').mkdir(parents=True, exist_ok=True)

for f in ['osp_best.pt', 'yolov8n_6ch.pt', 'train_metrics.json']:
    src = Path('model/artifacts') / f
    if src.exists(): shutil.copy2(src, OUT / 'model' / 'artifacts' / f)

shutil.copy2(TILES / 'prep_manifest.json', OUT / 'prep_manifest.json')
shutil.copy2(TILES / 'dataset.yaml',       OUT / 'dataset.yaml')

# Held-out validation split, for local INT8 calibration and re-scoring.
shutil.copytree(TILES / 'images' / 'val', OUT / 'val' / 'images')
shutil.copytree(TILES / 'labels' / 'val', OUT / 'val' / 'labels')

archive = shutil.make_archive(str(WORK / 'osp_dota_artifacts'), 'zip', str(OUT))
shutil.rmtree(OUT)
print('wrote', archive, f'({os.path.getsize(archive)/1e6:.1f} MB)')
print('\nDownload osp_dota_artifacts.zip from the Output panel on the right.')
